# wesep 논문 Table 2 — 조건화 4종 config 만들기

[wesep](https://github.com/wenet-e2e/wesep) 논문 Table 2 는 **BSRNN 백본에 조건화 방식(`spk_fuse_type`)만 바꾼 4 run** 임.
저장소 기본 [confs/bsrnn.yaml](../examples/librimix/tse/v2/confs/bsrnn.yaml) 은 논문 조건과 **화자 인코더가 다르므로**(ResNet34 를 처음부터 joint 학습) 그것도 함께 바꿔야 함.

이 노트북이 하는 일:

| # | 하는 일 |
|---|---|
| 1 | 기본 config 를 읽어 **무엇이 들어 있는지** 표로 |
| 2 | `make_fusion_config()` 함수로 **4벌 생성** |
| 3 | **전 / 후** 를 DataFrame 으로 대조 |
| 4 | 바꾸기로 한 키 말고는 **안 바뀌었는지** 검증 |
| 5 | **원본 yaml 이 얼마나 그대로 남았는지** — 주석·앵커·줄 수와 원문 diff |
| 6 | 실제 모델을 만들어 **fusion 층 파라미터 수**를 세고 논문 표와 대조 |

| 항목 | 내용 |
|---|---|
| **커널** | **`wesep2`** — py3.9 · torch 2.7.1+cu128. `wesep` · `wespeaker` 가 이 env 에 있음 |
| **경로 기준** | `wesep/__init__.py` 를 담은 폴더를 **위로 올라가며 찾음.**<br>바깥 저장소(SD-FiLM)의 `rootutils` · `.project-root` 에 기대지 않음 — **wesep 단독으로 돌아야 하기 때문** |
| **yaml 을 다루는 법** | **`ruamel.yaml` 왕복** — 앵커(`&embed_dim`)·별칭(`*speaker_feat`)·주석·빈 줄을 살린 채 값만 바꿈.<br>`pyyaml` 은 읽는 순간 이 셋을 전부 버려서 쓸 수 없음 |
| 건드리지 않은 것 | [confs/bsrnn.yaml](../examples/librimix/tse/v2/confs/bsrnn.yaml) 원본(읽기만) · wesep 소스 0줄 · `data/clean/`(stage 1·2 산출물, 안 읽음) |

세팅 절차 전체는 [wesep2_setup.md](../../docs/ops/wesep2_setup.md), 조사 경위는 [#72](../../docs/issues/wesep_fusion_reproduction.md) 에 있음 — 둘 다 바깥 저장소(SD-FiLM) 쪽 문서임.

In [1]:
import os, sys, io, re, copy, difflib, contextlib, platform
from pathlib import Path


def find_wesep_root(start: Path) -> Path:
    """`wesep/__init__.py` 를 담고 있는 폴더를 위로 올라가며 찾음 — 그것이 wesep 클론 루트임."""
    for d in (start, *start.parents):
        if (d / "wesep" / "__init__.py").is_file():
            return d
    raise FileNotFoundError(f"{start} 위쪽에 wesep 클론 루트가 없음")


WESEP  = find_wesep_root(Path.cwd())            # <wesep>                — import wesep 이 찾는 곳
RECIPE = WESEP / "examples/librimix/tse/v2"     # <wesep>/examples/…/v2  — run.sh · confs/ 가 있는 곳
BASE   = RECIPE / "confs/bsrnn.yaml"            # 손대지 않는 기본 config

if str(WESEP) not in sys.path:                  # 노트북 폴더에서 열려도 import wesep 이 되게
    sys.path.insert(0, str(WESEP))

import yaml, torch, pandas as pd                # yaml(pyyaml) 은 검증에서 읽기 전용으로만 씀
from ruamel.yaml import YAML
from ruamel.yaml.scalarint import ScalarInt
from ruamel.yaml.tokens import CommentToken
from ruamel.yaml.error import CommentMark
import wesep, wespeaker
from wesep.models.bsrnn import BSRNN
from IPython.display import display

PD_DISPLAY = {                  # 표가 잘려 보이지 않게
    "display.width":        220,
    "display.max_columns":   40,
    "display.max_colwidth":  90,
}
for opt, value in PD_DISPLAY.items():
    pd.set_option(opt, value)

FUSIONS = ["concat", "additive", "multiply", "FiLM"]   # Table 2 에서 run 마다 갈리는 것 — 이것 하나뿐

TABLE2_SHARED = {          # Table 2 의 4 run 이 공유하는 조건 — 저장소 기본값을 덮어씀. 키 순서는 bsrnn.yaml 과 같게
    "spk_model":        "ECAPA_TDNN_GLOB_c512",
    "spk_model_init":   "./wespeaker_models/voxceleb_ECAPA512/avg_model.pt",
    "spk_args":         {"feat_dim": 80, "embed_dim": 192, "pooling_func": "ASTP"},
                        # two_emb_layer 는 안 넣음 — ResNet34 전용 인자라 ECAPA 에 주면 TypeError
    "spk_emb_dim":      192,                    # yaml 에서는 *embed_dim 별칭으로 나감 (값을 두 번 안 적음)
    "spk_model_freeze": True,
}

TABLE2_TOP = {             # model_args 밖에서 바꾸는 것
    "gpus": "0",           # 기본값 '0,1' 은 죽은 값 — run.sh 가 --gpus "[0]" 로 덮고, 문자열이라 2장이면 rank 1 이 죽음
    "save_epoch_interval": 10,  # 10 epoch 마다 한 판 — 도중에 죽어도 여기서 재개
    "keep_last_epochs":    20,  # 그리고 마지막 20 epoch. 체크포인트 1개가 270 MB 라 150개 다 두면 39.5 GB
}

display(pd.DataFrame([
    {"항목": "python",      "값": platform.python_version(),           "위치": sys.executable},
    {"항목": "torch",       "값": torch.__version__,                   "위치": Path(torch.__file__).parent},
    {"항목": "ruamel.yaml", "값": "round-trip 사용",                    "위치": Path(YAML.__module__ and __import__("ruamel.yaml").yaml.__file__).parent},
    {"항목": "wesep",       "값": "import OK (BSRNN 포함)",             "위치": Path(wesep.__file__).parent},
    {"항목": "wespeaker",   "값": "import OK",                         "위치": Path(wespeaker.__file__).parent},
    {"항목": "wesep 루트",   "값": "찾음",                               "위치": WESEP},
    {"항목": "기본 config",  "값": "있음" if BASE.exists() else "없음",    "위치": BASE.relative_to(WESEP)},
]))

/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/kaldiio/__init__.py:2: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/s3prl/upstream/byol_s/byol_a/common.py:20: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")
ESPnet is not installed, cannot use espnet_hubert upstream


/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,항목,값,위치
0,python,3.9.25,/root/miniconda3/envs/wesep2/bin/python
1,torch,2.7.1+cu128,/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/torch
2,ruamel.yaml,round-trip 사용,/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/ruamel/yaml
3,wesep,import OK (BSRNN 포함),/workspace/git_clone/SD-FiLM/wesep/wesep
4,wespeaker,import OK,/root/miniconda3/envs/wesep2/lib/python3.9/site-packages/wespeaker
5,wesep 루트,찾음,/workspace/git_clone/SD-FiLM/wesep
6,기본 config,있음,examples/librimix/tse/v2/confs/bsrnn.yaml


## 1. 기본 config 에 무엇이 들어 있나

논문 조건과 갈리는 것은 `model_args.tse_model` 안의 **화자 인코더 관련 키**임.

In [2]:
base = yaml.safe_load(open(BASE))
m = base["model_args"]["tse_model"]

key_conds = [
    ("spk_fuse_type",    "4종으로 갈림"),
    *TABLE2_SHARED.items(),                                     # 바꾸는 값은 TABLE2_SHARED 한 곳에만 적혀 있음
    ("joint_training",   "True 유지 (spk_model_freeze 로 동결 여부를 정함)"),
    ("multi_fuse",       "False 유지 (fusion 을 1회만)"),
    ("use_spk_transform","False 유지"),
]
rows = [{"키": k, "기본값": m.get(k), "논문 Table 2 조건": v} for k, v in key_conds]
display(pd.DataFrame(rows))

,키,기본값,논문 Table 2 조건
0,spk_fuse_type,multiply,4종으로 갈림
1,spk_model,ResNet34,ECAPA_TDNN_GLOB_c512
2,spk_model_init,False,./wespeaker_models/voxceleb_ECAPA512/avg_model.pt
3,spk_args,"{'feat_dim': 80, 'embed_dim': 256, 'pooling_func': 'TSTP', 'two_emb_layer': False}","{'feat_dim': 80, 'embed_dim': 192, 'pooling_func': 'ASTP'}"
4,spk_emb_dim,256,192
5,spk_model_freeze,False,True
6,joint_training,True,True 유지 (spk_model_freeze 로 동결 여부를 정함)
7,multi_fuse,False,False 유지 (fusion 을 1회만)
8,use_spk_transform,False,False 유지


## 2. fusion config 4벌 만들기

바꾸는 키는 **6개뿐** — `TABLE2_SHARED` 의 5개와 `spk_fuse_type` 임.
나머지(`dataset_args` · `optimizer` · `scheduler` · `num_epochs` …)는 기본값 그대로 둠 — 네 run 이 조건화 방식 하나만 달라야 비교가 성립하기 때문임.

**같은 값을 두 번 적지 않음** — 위 1번 표도, 아래 4번의 `INTENDED` 도 전부 `TABLE2_SHARED` 에서 나옴.

`ruamel.yaml` 왕복이라 **원본 파일의 모양이 거의 그대로 남음.** 다만 왕복만 해도 손이 가는 곳이 넷 있어서 함수 안에서 되돌림:

| 손이 가는 곳 | 되돌리는 법 |
|---|---|
| `null` 이 빈 값으로 바뀜 | `add_representer(type(None), …)` |
| `False` · `True` 의 대소문자가 뭉개짐 | 원본 줄의 표기를 그대로 다시 씀 |
| `*별칭` 뒤의 꼬리 주석이 다음 줄로 내려감 | 별칭 줄만 겨냥한 정규식으로 도로 붙임 |
| 키를 지우면 거기 달린 주석 블록이 사라짐 | 지우기 전에 **앞 키의 꼬리 주석에 이어 붙임** |

In [3]:
BOOL_LINE = re.compile(r"^(\s*)([\w.-]+):(\s+)((?:&[\w-]+ )?)(True|False|true|false)(\b.*)$")


def _round_trip_yaml() -> YAML:
    """앵커·주석·빈 줄을 살리는 round-trip 로더. null 을 빈 값으로 바꾸지 않게 해 둠."""
    y = YAML()
    y.preserve_quotes = True
    y.width = 4096                                   # 긴 주석 줄이 접히지 않게
    y.representer.add_representer(
        type(None), lambda r, d: r.represent_scalar("tag:yaml.org,2002:null", "null"))
    return y


def _set_tail_comment(node, key: str, text: str, col: int) -> None:
    """그 키 줄 끝의 주석을 갈아 끼움 — 원본 주석이 내용과 어긋나게 될 때만 씀."""
    node.ca.items[key] = [None, None, CommentToken(f"# {text}\n", CommentMark(col)), None]


def _restore_bool_spelling(out: str, src: str, changed: dict) -> str:
    """ruamel 이 소문자로 통일해 버린 불린을 원본 줄의 표기로 되돌림."""
    orig = {(g[1], g[2]): g[5] for l in src.splitlines() if (g := BOOL_LINE.match(l))}
    orig.update(changed)                             # 우리가 바꾼 값은 파일의 다수 표기를 따름
    def fix(l):
        g = BOOL_LINE.match(l)
        if g and (k := (g[1], g[2])) in orig:
            return f"{g[1]}{g[2]}:{g[3]}{g[4]}{orig[k]}{g[6]}"
        return l
    return "\n".join(fix(l) for l in out.splitlines()) + "\n"


def make_fusion_config(fuse: str, base_path: Path = BASE,
                       shared: dict = TABLE2_SHARED, top: dict = TABLE2_TOP) -> str:
    """기본 config 를 ruamel 왕복으로 읽어 `shared` 조건 + 주어진 fusion 으로 바꾼 **yaml 텍스트**를 돌려줌.

    바꾸는 키 = model_args 안쪽 `shared` 5개 + `spk_fuse_type` 1개, model_args 밖 `top` 1개.
    앵커 `&embed_dim` 과 별칭 `*embed_dim` · `*speaker_feat` 은 그대로 유지됨.
    """
    src = base_path.read_text()
    y   = _round_trip_yaml()
    cfg = y.load(io.StringIO(src))
    m   = cfg["model_args"]["tse_model"]
    sa  = m["spk_args"]

    # 1) ECAPA 에 없는 two_emb_layer 를 지움. 그 키에 달려 있던 주석 블록(ECAPA·CAMPPlus 예시)은 앞 키 꼬리로 옮김
    if "two_emb_layer" in sa:
        tail = sa.ca.items["pooling_func"][2]
        tail.value = tail.value.rstrip("\n") + sa.ca.items.pop("two_emb_layer")[2].value
        del sa["two_emb_layer"]

    # 2) spk_args 를 제자리 수정 — 키 순서가 유지되고, embed_dim 에 앵커를 달아 spk_emb_dim 이 그것을 가리키게 함
    assert shared["spk_args"]["embed_dim"] == shared["spk_emb_dim"], "embed_dim 과 spk_emb_dim 이 어긋남"
    embed = ScalarInt(shared["spk_args"]["embed_dim"], anchor="embed_dim")
    sa["feat_dim"]     = shared["spk_args"]["feat_dim"]
    sa["embed_dim"]    = embed
    sa["pooling_func"] = shared["spk_args"]["pooling_func"]
    m["spk_emb_dim"]   = embed                       # 같은 객체라 *embed_dim 별칭으로 나감

    # 3) 나머지 키
    for k, v in shared.items():
        if k not in ("spk_args", "spk_emb_dim"):
            m[k] = v
    m["spk_fuse_type"] = fuse
    for k, v in top.items():
        cfg[k] = v

    # 4) 화자 인코더가 바뀌어 내용이 틀려진 꼬리 주석 두 개를 갈아 끼움
    _set_tail_comment(m, "spk_model", "ECAPA_TDNN_c512/c1024, ECAPA_TDNN_GLOB_c512/c1024", 40)
    _set_tail_comment(m, "spk_model_init", "stage 1 이 wespeaker_models/ 로 받아 둠", 60)

    buf = io.StringIO(); y.dump(cfg, buf); out = buf.getvalue()
    out = re.sub(r"(:[ \t]*\*[\w-]+)\n[ \t]+(#[^\n]*)", r"\1   \2", out)   # 별칭 뒤 꼬리 주석을 도로 앞 줄에
    return _restore_bool_spelling(out, src, {("    ", "spk_model_freeze"): "True"})


fusion_configs, rows = [], []    # fusion_configs 는 뒤 셀들이 쓰는 데이터, rows 는 이 셀의 표
for fusion in FUSIONS:
    new_text = make_fusion_config(fusion)
    new_path = RECIPE / f"confs/bsrnn_ecapa_{fusion}.yaml"
    new_path.write_text(new_text)
    new_cfg  = yaml.safe_load(io.StringIO(new_text))
    fusion_configs.append({"fusion": fusion, "new_path": new_path,
                           "new_cfg": new_cfg, "new_text": new_text})
    rows.append({"fusion": fusion,
                 "파일": str(new_path.relative_to(WESEP)),
                 "크기(B)": new_path.stat().st_size})

display(pd.DataFrame(rows))

,fusion,파일,크기(B)
0,concat,examples/librimix/tse/v2/confs/bsrnn_ecapa_concat.yaml,3688
1,additive,examples/librimix/tse/v2/confs/bsrnn_ecapa_additive.yaml,3690
2,multiply,examples/librimix/tse/v2/confs/bsrnn_ecapa_multiply.yaml,3690
3,FiLM,examples/librimix/tse/v2/confs/bsrnn_ecapa_FiLM.yaml,3686


## 3. 전 / 후 대조

`model_args.tse_model` 의 키를 전부 훑어 **기본값과 달라진 것만** 표로. 네 벌이 서로 `spk_fuse_type` 하나만 달라야 함.

In [4]:
base_m = yaml.safe_load(open(BASE))["model_args"]["tse_model"]

rows = []
for fusion_config in fusion_configs:
    fusion, new_cfg = fusion_config["fusion"], fusion_config["new_cfg"]
    new_m = new_cfg["model_args"]["tse_model"]
    for k in sorted(set(base_m) | set(new_m)):
        b, a = base_m.get(k, "<없음>"), new_m.get(k, "<없음>")
        if b != a:
            rows.append({"fusion": fusion, "키": k, "전 (bsrnn.yaml)": b, "후": a})

df_diff = pd.DataFrame(rows)
display(df_diff)

cnt = df_diff.groupby("fusion").size().rename("바뀐 키 개수").reset_index()
cnt["비고"] = cnt["fusion"].map(lambda f: "기본값이 이미 multiply 라 1개 적음" if f == "multiply" else "")
display(cnt)

,fusion,키,전 (bsrnn.yaml),후
0,concat,spk_args,"{'feat_dim': 80, 'embed_dim': 256, 'pooling_func': 'TSTP', 'two_emb_layer': False}","{'feat_dim': 80, 'embed_dim': 192, 'pooling_func': 'ASTP'}"
1,concat,spk_emb_dim,256,192
2,concat,spk_fuse_type,multiply,concat
3,concat,spk_model,ResNet34,ECAPA_TDNN_GLOB_c512
4,concat,spk_model_freeze,False,True
5,concat,spk_model_init,False,./wespeaker_models/voxceleb_ECAPA512/avg_model.pt
6,additive,spk_args,"{'feat_dim': 80, 'embed_dim': 256, 'pooling_func': 'TSTP', 'two_emb_layer': False}","{'feat_dim': 80, 'embed_dim': 192, 'pooling_func': 'ASTP'}"
7,additive,spk_emb_dim,256,192
8,additive,spk_fuse_type,multiply,additive
9,additive,spk_model,ResNet34,ECAPA_TDNN_GLOB_c512


,fusion,바뀐 키 개수,비고
0,FiLM,6,
1,additive,6,
2,concat,6,
3,multiply,5,기본값이 이미 multiply 라 1개 적음


## 4. 나머지는 안 바뀌었는지 검증

**바꾸기로 한 키 말고는 전부 같아야 함.** 최상위 블록(`dataset_args` · `dataloader_args` · `optimizer_args` · `scheduler_args` …)을 통째로 비교함.
`TABLE2_TOP` 의 `gpus` · `keep_last_epochs` 는 의도해서 바꾼 것이라 비교에서 빼고 값만 열로 보여 줌.
`keep_last_epochs` 는 **저장소 원본에 없던 키**임 — [train.py:371-372](../wesep/bin/train.py#L371-L372) 가 `configs.get()` 으로 읽어, 없으면 전부 저장하던 기존 동작 그대로임.

In [5]:
INTENDED = {"spk_fuse_type", *TABLE2_SHARED}   # 손으로 다시 안 적음 — 상수에서 유도하므로 어긋날 수 없음

rows = []
base_all = yaml.safe_load(open(BASE))
for fusion_config in fusion_configs:
    fusion, new_cfg = fusion_config["fusion"], fusion_config["new_cfg"]
    top_same = all(new_cfg.get(k) == base_all.get(k)
                   for k in set(base_all) | set(new_cfg) if k not in {"model_args", *TABLE2_TOP})
    changed = {k for k in set(base_m) | set(new_cfg["model_args"]["tse_model"])
               if base_m.get(k, "<없음>") != new_cfg["model_args"]["tse_model"].get(k, "<없음>")}
    rows.append({
        "fusion": fusion,
        "model_args 밖이 동일": top_same,     # TABLE2_TOP 의 키는 빼고 비교
        "gpus": new_cfg["gpus"],
        "save_epoch_interval": new_cfg["save_epoch_interval"],
        "keep_last_epochs": new_cfg["keep_last_epochs"],
        "바뀐 키": sorted(changed),
        "의도한 키만 바뀜": changed <= INTENDED,
        "num_epochs": new_cfg["num_epochs"],
        "batch_size": new_cfg["dataloader_args"]["batch_size"],
        "chunk_len": new_cfg["dataset_args"]["chunk_len"],
        "loss": new_cfg["loss"],
    })
display(pd.DataFrame(rows))

,fusion,model_args 밖이 동일,gpus,save_epoch_interval,keep_last_epochs,바뀐 키,의도한 키만 바뀜,num_epochs,batch_size,chunk_len,loss
0,concat,True,0,10,20,"[spk_args, spk_emb_dim, spk_fuse_type, spk_model, spk_model_freeze, spk_model_init]",True,150,8,48000,SISDR
1,additive,True,0,10,20,"[spk_args, spk_emb_dim, spk_fuse_type, spk_model, spk_model_freeze, spk_model_init]",True,150,8,48000,SISDR
2,multiply,True,0,10,20,"[spk_args, spk_emb_dim, spk_model, spk_model_freeze, spk_model_init]",True,150,8,48000,SISDR
3,FiLM,True,0,10,20,"[spk_args, spk_emb_dim, spk_fuse_type, spk_model, spk_model_freeze, spk_model_init]",True,150,8,48000,SISDR


## 5. 원본 yaml 이 얼마나 남았나

값이 맞는 것과 **파일이 읽을 만한 것**은 다른 문제임. 주석·앵커·빈 줄이 살아 있어야 다음에 열었을 때 무엇이 왜 그런지 알 수 있음.
아래 표로 세고, 그 다음에 **원문 diff** 를 그대로 찍어 바뀐 줄이 전부 의도한 것인지 눈으로 확인함.

In [6]:
src = BASE.read_text()
count = lambda t: {"줄": len(t.splitlines()), "주석(#)": t.count("#"),
                   "앵커(&)": t.count("&"), "별칭(*)": t.count("*")}

rows = [{"대상": "원본 bsrnn.yaml", **count(src), "비고": ""}]
for fusion_config in fusion_configs:
    rows.append({"대상": fusion_config["fusion"], **count(fusion_config["new_text"]),
                 "비고": "two_emb_layer 1줄 삭제"})
display(pd.DataFrame(rows))

print("=== bsrnn.yaml → bsrnn_ecapa_FiLM.yaml 원문 diff ===")
new_text = next(c["new_text"] for c in fusion_configs if c["fusion"] == "FiLM")
for line in difflib.unified_diff(src.splitlines(), new_text.splitlines(),
                                 "bsrnn.yaml", "bsrnn_ecapa_FiLM.yaml", lineterm="", n=1):
    print(line)

,대상,줄,주석(#),앵커(&),별칭(*),비고
0,원본 bsrnn.yaml,116,126,6,3,
1,concat,116,126,6,3,two_emb_layer 1줄 삭제
2,additive,116,126,6,3,two_emb_layer 1줄 삭제
3,multiply,116,126,6,3,two_emb_layer 1줄 삭제
4,FiLM,116,126,6,3,two_emb_layer 1줄 삭제


=== bsrnn.yaml → bsrnn_ecapa_FiLM.yaml 원문 diff ===
--- bsrnn.yaml
+++ bsrnn_ecapa_FiLM.yaml
@@ -28,3 +28,3 @@
   # if you want to use multi-optimization, please ref bsrnn_multi_optim.yaml
-  SSA_enroll_prob:  0  # prob to add SSA on enrollment speech
+  SSA_enroll_prob: 0   # prob to add SSA on enrollment speech
 
@@ -32,3 +32,3 @@
 exp_dir: exp/BSRNN
-gpus: '0,1'
+gpus: '0'
 log_batch_interval: 100
@@ -36,3 +36,3 @@
 loss: SISDR
-loss_args: { }
+loss_args: {}
 
@@ -52,3 +52,3 @@
     num_repeat: 6
-    spk_fuse_type: 'multiply'
+    spk_fuse_type: 'FiLM'
     use_spk_transform: False
@@ -57,9 +57,8 @@
     ####### ResNet    The pretrained speaker encoders are available from: https://github.com/wenet-e2e/wespeaker/blob/master/docs/pretrained.md
-    spk_model: ResNet34 # ResNet18, ResNet34, ResNet50, ResNet101, ResNet152
-    spk_model_init: False #./wespeaker_models/voxceleb_resnet34/avg_model.pt
+    spk_model: ECAPA_TDNN_GLOB_c512     # ECAPA_TDNN_c512/c1024, ECAPA_TDNN_GLOB_c512/c1

## 6. fusion 층 파라미터 수 실측

config 만으로는 *"정말 4종이 다른 층을 만드는가"* 를 알 수 없으므로, **실제로 `BSRNN` 을 만들어** `separator.separation[0]`(= `SpeakerFuseLayer`) 의 파라미터를 셈.
사전학습 가중치도 이때 함께 로드되므로 **`spk_model_init` 경로가 맞는지도 여기서 드러남.**

손계산과 대조 — `e=192` · `c=128` 일 때
`concat` = (192+128)·128+128 = **41,088** · `additive`·`multiply` = 192·128+128 = **24,704** · `FiLM` = 2×24,704 = **49,408**.

`spk_model_init` 이 `./wespeaker_models/…` 인 **상대경로**라 레시피 폴더에서 만들어야 함 — 그 안에서만 `os.chdir` 함.

In [7]:
@contextlib.contextmanager
def in_dir(d: Path):
    """레시피 폴더에서만 모델을 만들고 원래 폴더로 돌아옴 (spk_model_init 이 상대경로라서)."""
    prev = Path.cwd()
    os.chdir(d)
    try:
        yield
    finally:
        os.chdir(prev)


HAND = {"concat": 41088, "additive": 24704, "multiply": 24704, "FiLM": 49408}
PAPER = {"concat": 12.84, "additive": 13.15, "multiply": 13.25, "FiLM": 13.32}

rows = []
with in_dir(RECIPE):
    for fusion_config in fusion_configs:
        fusion, new_path = fusion_config["fusion"], fusion_config["new_path"]
        model_args = yaml.safe_load(open(new_path))["model_args"]["tse_model"]
        model = BSRNN(**model_args)
        fl = model.separator.separation[0]
        rows.append({
            "fusion": fusion,
            "fuse_layer": type(fl).__name__,
            "fc": type(fl.fc).__name__,
            "fuse_params": f"{sum(x.numel() for x in fl.parameters()):,}",
            "손계산": HAND[fusion],
            "일치": sum(x.numel() for x in fl.parameters()) == HAND[fusion],
            "전체": f"{sum(x.numel() for x in model.parameters()):,}",
            "학습대상": f"{sum(x.numel() for x in model.parameters() if x.requires_grad):,}",
            "spk 동결": all(not x.requires_grad for x in model.spk_model.parameters()),
            "논문 SI-SDR": PAPER[fusion],
        })
        del model

display(pd.DataFrame(rows))

,fusion,fuse_layer,fc,fuse_params,손계산,일치,전체,학습대상,spk 동결,논문 SI-SDR
0,concat,SpeakerFuseLayer,LinearLayer,"41,088",41088,True,"27,634,184","21,443,464",True,12.84
1,additive,SpeakerFuseLayer,LinearLayer,"24,704",24704,True,"27,617,800","21,427,080",True,13.15
2,multiply,SpeakerFuseLayer,LinearLayer,"24,704",24704,True,"27,617,800","21,427,080",True,13.25
3,FiLM,SpeakerFuseLayer,FiLM,"49,408",49408,True,"27,642,504","21,451,784",True,13.32


## 정리

| 확인한 것 | 결과 |
|---|---|
| config 4벌 생성 | `confs/bsrnn_ecapa_{concat,additive,multiply,FiLM}.yaml` |
| 바뀐 키 | fusion 당 **6개** — `spk_fuse_type` · `spk_model` · `spk_args` · `spk_emb_dim` · `spk_model_init` · `spk_model_freeze` |
| 나머지 동일 | `dataset_args` · `dataloader_args` · `optimizer_args` · `scheduler_args` · `num_epochs` · `loss` 전부 기본값 |
| 원본 모양 | 주석·앵커·별칭·빈 줄 보존. 줄 수는 `two_emb_layer` 1줄만 줄어듦 |
| fusion 층 | 4종이 서로 다른 파라미터 수를 가지며 손계산과 일치 |
| 화자 인코더 | 사전학습 가중치 로드 성공 + 전부 `requires_grad=False` |

다음은 [wesep2_setup.md](../../docs/ops/wesep2_setup.md) 의 **4-b** — 이 config 4벌로 stage 3~6 을 run 마다 돌림.